# SolveX 3.0: Data Cleaning, Normalization & Preprocessing Pipeline

This notebook inspects the raw 2024 Algerian road safety datasets, cleans data quality anomalies (subtotals, unit mismatches, dimension mixing), conducts exploratory data analysis (EDA), generates a calibrated incident-level training dataset, and normalizes features for model training.

> **Note:** Model training has been decoupled into `train.ipynb`.


In [ ]:
import os
import glob
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

DATA_DIR = os.path.join(os.getcwd(), 'data')
CLEANED_DIR = os.path.join(DATA_DIR, 'cleaned')
os.makedirs(CLEANED_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Cleaned output directory: {CLEANED_DIR}")


## 1. Inspecting Raw Datasets & Identifying Data Anomalies


In [ ]:
raw_files = {
    'causes': pd.read_csv(os.path.join(DATA_DIR, 'causes.csv')),
    'national_trend': pd.read_csv(os.path.join(DATA_DIR, 'national_trend.csv')),
    'rural_roads': pd.read_csv(os.path.join(DATA_DIR, 'rural_road_types.csv')),
    'time_dist': pd.read_csv(os.path.join(DATA_DIR, 'time_distribution.csv')),
    'vehicles': pd.read_csv(os.path.join(DATA_DIR, 'vehicle_types.csv')),
    'victim_demographics': pd.read_csv(os.path.join(DATA_DIR, 'victim_age_sex.csv')),
    'wilaya_top10': pd.read_csv(os.path.join(DATA_DIR, 'wilaya_top10.csv')),
    'supabase_demo': pd.read_csv(os.path.join(DATA_DIR, 'supabase_incidents_demo_rows.csv'))
}

print("=== Raw Dataset Summary ===")
for name, df in raw_files.items():
    print(f"• {name:<20}: {df.shape[0]:>3} rows, {df.shape[1]:>2} cols | Nulls: {df.isnull().sum().sum()}")


## 2. Data Cleaning & Unit Harmonization

1. **`causes.csv`:** Filter subtotal contamination and recalculate mathematical shares.
2. **`time_distribution.csv`:** Decouple Time-of-Day from Weekday dimensions.
3. **`vehicle_types.csv`:** Exclude summary totals and recalculate distribution.
4. **`wilaya_top10.csv`:** Pivot metrics and calculate Lethality Index per Wilaya.


In [ ]:
# 1. Clean causes.csv
df_causes = raw_files['causes'].copy()
is_subtotal = df_causes['factor_group'].str.contains('subtotal|All groups', case=False, na=False)
is_total = df_causes['cause_original_or_translated'].str.contains('total', case=False, na=False)
df_causes_clean = df_causes[~(is_subtotal | is_total)].copy()

TOTAL_NATIONAL_ACCIDENTS = 24829
df_causes_clean['count'] = df_causes_clean['count'].astype(int)
df_causes_clean['share_ratio'] = df_causes_clean['count'] / TOTAL_NATIONAL_ACCIDENTS
df_causes_clean['share_percent'] = (df_causes_clean['share_ratio'] * 100).round(3)
df_causes_clean = df_causes_clean.drop(columns=['source_page', 'notes'], errors='ignore')
df_causes_clean = df_causes_clean.sort_values(by='count', ascending=False).reset_index(drop=True)

df_causes_clean.to_csv(os.path.join(CLEANED_DIR, 'causes_cleaned.csv'), index=False)
print(f"Cleaned causes: {len(df_causes_clean)} categories saved.")
display(df_causes_clean.head(8))


In [ ]:
# 2. Decouple time_distribution.csv into Time of Day and Weekday tables
df_time = raw_files['time_dist'].copy()
df_time['category'] = df_time['category'].str.replace('–', '-')

df_time_of_day = df_time[df_time['dimension'] == 'Time of day'].copy().drop(columns=['dimension', 'source_page'])
total_tod = df_time_of_day['accident_count'].sum()
df_time_of_day['share_ratio'] = df_time_of_day['accident_count'] / total_tod
df_time_of_day['share_percent'] = (df_time_of_day['share_ratio'] * 100).round(2)
df_time_of_day.to_csv(os.path.join(CLEANED_DIR, 'time_of_day_cleaned.csv'), index=False)

df_weekday = df_time[df_time['dimension'] == 'Weekday'].copy().drop(columns=['dimension', 'source_page'])
total_wd = df_weekday['accident_count'].sum()
df_weekday['share_ratio'] = df_weekday['accident_count'] / total_wd
df_weekday['share_percent'] = (df_weekday['share_ratio'] * 100).round(2)
df_weekday.to_csv(os.path.join(CLEANED_DIR, 'weekday_cleaned.csv'), index=False)

print("Decoupled temporal tables:")
display(df_time_of_day)
display(df_weekday)


In [ ]:
# 3. Clean vehicles, road types, and Wilaya statistics
df_veh = raw_files['vehicles'].copy()
df_veh_clean = df_veh[df_veh['vehicle_category'].str.lower() != 'total'].copy()
total_veh = df_veh_clean['count'].sum()
df_veh_clean['share_ratio'] = df_veh_clean['count'] / total_veh
df_veh_clean['share_percent'] = (df_veh_clean['share_ratio'] * 100).round(2)
df_veh_clean = df_veh_clean.drop(columns=['source_page', 'notes'], errors='ignore')
df_veh_clean.to_csv(os.path.join(CLEANED_DIR, 'vehicle_types_cleaned.csv'), index=False)

df_roads = raw_files['rural_roads'].copy()
df_roads['road_type'] = df_roads['road_type'].str.replace('–', '-')
total_roads = df_roads['accidents_2024'].sum()
df_roads['share_ratio'] = df_roads['accidents_2024'] / total_roads
df_roads['share_percent'] = (df_roads['share_ratio'] * 100).round(2)
df_roads = df_roads.drop(columns=['source_page', 'notes'], errors='ignore')
df_roads.to_csv(os.path.join(CLEANED_DIR, 'road_types_cleaned.csv'), index=False)

df_w = raw_files['wilaya_top10'].copy()
df_w['wilaya'] = df_w['wilaya'].str.replace('Algiers', 'Alger').str.replace('Setif', 'Sétif')
df_w_pivot = df_w.pivot_table(index='wilaya', columns='metric', values='count', aggfunc='sum', fill_value=0).reset_index()
df_w_pivot.columns.name = None
df_w_pivot = df_w_pivot.rename(columns={'Accidents': 'accidents', 'Deaths': 'deaths', 'Injuries': 'injuries'})
df_w_pivot['lethality_rate_pct'] = ((df_w_pivot['deaths'] / df_w_pivot['accidents']) * 100).round(2)
df_w_pivot = df_w_pivot.sort_values(by=['accidents', 'deaths'], ascending=False).reset_index(drop=True)
df_w_pivot.to_csv(os.path.join(CLEANED_DIR, 'wilaya_risk_cleaned.csv'), index=False)

print("Saved cleaned vehicle types, road types, and Wilaya profiles.")


## 3. Exploratory Data Analysis & Empirical Visualizations


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Top 8 Causes
top_causes = df_causes_clean.head(8)
sns.barplot(data=top_causes, y='cause_original_or_translated', x='share_percent', hue='factor_group', dodge=False, ax=axes[0, 0], palette='Reds_r')
axes[0, 0].set_title('Top 8 Accident Causes in Algeria (2024)', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Share of Total Accidents (%)')
axes[0, 0].set_ylabel('')

# Plot 2: Time of Day
sns.barplot(data=df_time_of_day, x='category', y='share_percent', ax=axes[0, 1], palette='Blues_r')
axes[0, 1].set_title('Accident Distribution by Time of Day', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Time Window')
axes[0, 1].set_ylabel('Percentage of Accidents (%)')
axes[0, 1].tick_params(axis='x', rotation=25)

# Plot 3: Urban vs. Rural Lethality
df_trend = raw_files['national_trend']
trend_sub = df_trend[df_trend['area'].isin(['Urban', 'Rural'])].pivot(index='area', columns='indicator', values='period_2024')
trend_sub['fatality_rate_pct'] = (trend_sub['deaths'] / trend_sub['accidents'] * 100).round(1)

sns.barplot(x=trend_sub.index, y=trend_sub['fatality_rate_pct'], ax=axes[1, 0], palette=['#2b5c8f', '#c0392b'])
axes[1, 0].set_title('Crash Lethality: Urban vs. Rural Roads (% Deaths/Accidents)', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Lethality Rate (%)')
for p in axes[1, 0].patches:
    axes[1, 0].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                        ha='center', va='center', color='white', fontweight='bold', fontsize=13)

# Plot 4: Top Wilayas by Deaths
top_deaths_w = df_w_pivot.sort_values(by='deaths', ascending=False).head(8)
sns.barplot(data=top_deaths_w, x='wilaya', y='deaths', ax=axes[1, 1], palette='Oranges_r')
axes[1, 1].set_title('Top 8 Wilayas by Recorded Deaths', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Wilaya')
axes[1, 1].set_ylabel('Annual Deaths')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


## 4. Synthesizing Calibrated Incident Data & Feature Preprocessing

We load the calibrated 10,000 incident samples (generated from the empirical distributions in `prepare_data.py`), perform feature engineering (cyclical hour $\sin$/$\cos$, one-hot encodings), scale continuous attributes, and export the preprocessed train/test matrices.


In [ ]:
training_data_path = os.path.join(DATA_DIR, 'incidents_training_dataset.csv')
df_incidents = pd.read_csv(training_data_path)

print(f"Calibrated Incident Dataset: {df_incidents.shape[0]:,} records across {df_incidents.shape[1]} features.")
display(df_incidents[['created_at', 'wilaya', 'road_type', 'vehicles_count', 'injured_count', 'severity_score', 'severity_tier']].head())

print("\nClass Distribution:")
print(df_incidents['severity_tier'].value_counts(normalize=True).round(3) * 100)


In [ ]:
# Verify preprocessed matrices for train.ipynb
train_path = os.path.join(CLEANED_DIR, 'train_dataset_preprocessed.csv')
test_path = os.path.join(CLEANED_DIR, 'test_dataset_preprocessed.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"[Ready] Train dataset: {train_df.shape[0]:,} rows, {train_df.shape[1]} columns -> {train_path}")
print(f"[Ready] Test dataset:  {test_df.shape[0]:,} rows, {test_df.shape[1]} columns -> {test_path}")
print("\nAll data cleaning and preprocessing complete! Proceed to `train.ipynb` for model training.")
